# 01. 検出モデルの学習と評価(Colab)

`scripts/prepare_dataset.py split` で作ったデータセット(`data/dataset/yolo/`)を zip にして Google Drive に置き、
このノートブックで YOLOv8 をファインチューニングする。

1. ランタイム > ランタイムのタイプを変更 > T4 GPU
2. 下の設定セルの `DRIVE_DIR` を、zip を置いた Drive のフォルダに合わせる
3. 上から順に実行する。学習結果(重み・学習曲線・評価結果)は Drive の `runs/` に保存される

In [2]:
DRIVE_DIR = '/content/drive/MyDrive/dataset/wbt'   # yolo.zip を置いたフォルダ
MODEL = 'yolov8s.pt'                        # n / s / m を比較する場合はここを変えて再実行
IMGSZ = 1280                                # ボールが小さいため 1280 以上にする
EPOCHS = 100
RUN_NAME = f'{MODEL.split(".")[0]}_{IMGSZ}'

In [3]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics
!git clone -q https://github.com/ashina1992/wheelchair-basketball-tracking.git /content/wbt || true
!pip install -q -e /content/wbt
!unzip -q -o {DRIVE_DIR}/yolo.zip -d /content/dataset
!sed -i 's#^path: .*#path: /content/dataset#' /content/dataset/data.yaml
!cat /content/dataset/data.yaml

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 11.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for wbt (pyproject.toml) ... done
path: /content/dataset
train: images/train
val: images/val
test: images/test
names:
  0: player
  1: ball
  2: rim
  3: referee


In [4]:
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(
    data='/content/dataset/data.yaml', epochs=EPOCHS, imgsz=IMGSZ, batch=8, device=0,
    project=f'{DRIVE_DIR}/runs', name=RUN_NAME, exist_ok=True,
    patience=30, cos_lr=True,
    # 放送映像は左右反転しても成り立つが、上下反転は成り立たない
    fliplr=0.5, flipud=0.0, mosaic=1.0, close_mosaic=10,
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.178 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, free

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a97a3bae430>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

## test セットでの評価

既製の COCO モデル(person→player、sports ball→ball に読み替え)と、学習したモデルを同じ方法で評価する。
結果の JSON は `docs/03_finetune.md` にまとめる。

In [5]:
best = f'{DRIVE_DIR}/runs/{RUN_NAME}/weights/best.pt'
!cd /content/wbt && python scripts/eval_detector.py /content/dataset --split test --imgsz {IMGSZ} \
    --weights {MODEL} {best} --out {DRIVE_DIR}/runs/{RUN_NAME}/eval_test.json

yolov8s.pt
  player   AP50=0.244 AP50-95=0.087 n_gt=174
  ball     AP50=0.004 AP50-95=0.003 n_gt=16
/content/drive/MyDrive/dataset/wbt/runs/yolov8s_1280/weights/best.pt
  player   AP50=0.950 AP50-95=0.767 n_gt=174
  ball     AP50=0.963 AP50-95=0.417 n_gt=16
  rim      AP50=1.000 AP50-95=0.582 n_gt=18
  referee  AP50=0.958 AP50-95=0.834 n_gt=49


In [6]:
# ultralytics 標準の評価(クラス別 mAP、混同行列、PR 曲線の画像が保存される)
metrics = YOLO(best).val(data='/content/dataset/data.yaml', split='test', imgsz=IMGSZ,
                         project=f'{DRIVE_DIR}/runs', name=f'{RUN_NAME}_test', exist_ok=True)
print(metrics.results_dict)

Ultralytics 8.4.178 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3425.2±385.3 MB/s, size: 627.2 KB)
val: Scanning /content/dataset/labels/test... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 1.9Kit/s 0.0s
val: New cache created: /content/dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.1s/it 2.2s
                   all         20        257      0.949      0.977      0.969      0.649
                player         20        174      0.927      0.977       0.97      0.787
                  ball         16         16       0.94       0.98      0.959      0.422
                   rim         18         18          1      0.991      0.995      0.573
               referee         20         49      0.928      0.959      0.953      0.815